In [1]:
import sys, os

In [2]:
ls

trials.ipynb


In [3]:
cd ..

/Users/gojuruakshith/PycharmProjects/SafeResponse_Engine


In [24]:
from dataclasses import dataclass
from pathlib import Path
@dataclass(frozen=True)
class UserQueryConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path


In [25]:
from src.saferesponse_engine.utils.common import read_yaml, create_directories
from src.saferesponse_engine.constants import CONFIG_FILE_PATH, PARAM_FILE_PATH, SCHEMA_FILE_PATH
from pathlib import Path

class ConfigurationManager:
    def __init__(self):
        self.config = read_yaml(CONFIG_FILE_PATH)
        self.params = read_yaml(PARAM_FILE_PATH)
        self.schema = read_yaml(SCHEMA_FILE_PATH)
        create_directories([Path(self.config.artifacts_root)])

    def get_user_query_config(self) -> UserQueryConfig:
        config = self.config.user_query

        root_dir = Path(config.root_dir)
        create_directories([root_dir])

        return UserQueryConfig(
            root_dir=root_dir,
            source_url=str(config.source_url),
            local_data_file=Path(config.local_data_file),
        )

In [26]:
import ssl
from urllib import request
from urllib.parse import urlparse

from src.saferesponse_engine import logger
from src.saferesponse_engine.utils.common import get_size

class UserQuery:
    def __init__(self, config: UserQueryConfig):
        self.config = config

    @staticmethod
    def _resolve_source_url(source_url: str) -> str:
        parsed = urlparse(source_url)
        if parsed.netloc == "github.com" and "/blob/" in parsed.path:
            owner, repo, _, branch, *file_parts = parsed.path.strip("/").split("/")
            return f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/{'/'.join(file_parts)}"
        return source_url


    def download_file(self):
         if not self.config.local_data_file.exists():
              self.config.local_data_file.parent.mkdir(parents=True, exist_ok=True)
              ctx = ssl.create_default_context()
              source_url = self._resolve_source_url(self.config.source_url)

              with request.urlopen(source_url, context=ctx) as r:
                 text = r.read().decode("utf-8")

              self.config.local_data_file.write_text(text, encoding="utf-8")
              logger.info(f"Downloaded file to: {self.config.local_data_file}")
         else:
             logger.info(f"File already exists of size: {get_size(self.config.local_data_file)}")


In [27]:
try:
    cm = ConfigurationManager()
    data_ingestion_config = cm.get_user_query_config()
    data_ingestion = UserQuery(config=data_ingestion_config)
    data_ingestion.download_file()

except Exception as e:
    raise e

[2026-04-18 16:54:45,906: INFO: common: YAML file loaded successfully: config/config.yaml]
[2026-04-18 16:54:45,908: INFO: common: YAML file loaded successfully: params.yaml]
[2026-04-18 16:54:45,909: INFO: common: YAML file loaded successfully: schema.yaml]
[2026-04-18 16:54:45,909: INFO: common: Created directory at: artifacts]
[2026-04-18 16:54:45,910: INFO: common: Created directory at: artifacts/user_query]
[2026-04-18 16:54:45,910: INFO: 986366041: File already exists of size: ~ 0 KB]
